<a href="https://colab.research.google.com/github/Arnob07Mondal/Compiler_Design/blob/main/CompilerDesignLab9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Write a Python program to implement a simple LL(1) Predictive Parser that validates
whether an input string matches the grammar.**

In [ ]:
from collections import defaultdict

def find_first(symbol):
    if symbol not in grammar:
        return {symbol}
    result = set()
    for production in grammar[symbol]:
        # Epsilon production
        if production == ['e']:
            result.add('e')
            continue
        for x in production:
            temp = find_first(x)
            result.update(temp - {'e'})
            if 'e' not in temp:
                break
        else:
            result.add('e')
    return result

def find_follow():
    follow = defaultdict(set)
    follow[start_symbol].add('$')
    changed = True
    while changed:
        changed = False
        for lhs in grammar:
            for production in grammar[lhs]:
                for i in range(len(production)):
                    symbol = production[i]
                    if symbol in grammar:
                        old = len(follow[symbol])
                        if i == len(production) - 1:
                            follow[symbol].update(follow[lhs])
                        else:
                            next_symbol = production[i + 1]
                            first_next = find_first(next_symbol)
                            follow[symbol].update(
                                first_next - {'e'}
                            )
                            if 'e' in first_next:
                                follow[symbol].update(
                                    follow[lhs]
                                )
                        if len(follow[symbol]) > old:
                            changed = True
    return follow

print("LL(1) PARSING TABLE GENERATOR")
print("------------------------------")
n = int(input("Enter number of productions: "))
grammar = {}
for i in range(n):
    production = input("Enter production: ")
    left, right = production.split("->")
    left = left.strip()
    alternatives = right.split("|")
    grammar[left] = []
    for alternative in alternatives:
        alternative = alternative.strip()
        alternative = alternative.replace(" ", "")
        grammar[left].append(list(alternative))

start_symbol = list(grammar.keys())[0]

first = {}
for non_terminal in grammar:
    first[non_terminal] = find_first(non_terminal)

follow = find_follow()

terminals = set()
for lhs in grammar:
    for production in grammar[lhs]:
        for symbol in production:
            if symbol not in grammar and symbol != 'e':
                terminals.add(symbol)
terminals.add('$')
terminals = sorted(terminals)

table = defaultdict(dict)
for lhs in grammar:
    for production in grammar[lhs]:
        first_production = set()
        for symbol in production:
            temp = find_first(symbol)
            first_production.update(temp - {'e'})
            if 'e' not in temp:
                break
        else:
            first_production.add('e')

        for terminal in first_production:
            if terminal != 'e':
                table[lhs][terminal] = production
        if 'e' in first_production:
            for terminal in follow[lhs]:
                table[lhs][terminal] = production

print("\nFIRST SETS")
print("----------")
for nt in grammar:
    print("FIRST(" + nt + ") =", first[nt])

print("\nFOLLOW SETS")
print("-----------")
for nt in grammar:
    print("FOLLOW(" + nt + ") =", follow[nt])

print("\nLL(1) PARSING TABLE")
print("-------------------")
print("NT/T".ljust(10), end="")
for terminal in terminals:
    print(terminal.center(15), end="")
print()
print("-" * (10 + 15 * len(terminals)))
for nt in grammar:
    print(nt.ljust(10), end="")
    for terminal in terminals:
        if terminal in table[nt]:
            production = "".join(table[nt][terminal])
            print(
                (nt + "->" + production).center(15),
                end=""
            )
        else:
            print("-".center(15), end="")
    print()

LL(1) PARSING TABLE GENERATOR
------------------------------
Enter number of productions: 3
Enter production: S -> (L)/a
Enter production: L -> SL'
Enter production: L' -> )SL' /EPSILON

FIRST SETS
----------
FIRST(S) = {'('}
FIRST(L) = {'('}
FIRST(L') = {')'}

FOLLOW SETS
-----------
FOLLOW(S) = {'I', '(', '$'}
FOLLOW(L) = {"'", 'O', ')'}
FOLLOW(L') = set()

LL(1) PARSING TABLE
-------------------
NT/T             $              '              (              )              /              E              I              N              O              P              a       
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
S                -              -           S->(L)/a          -              -              -              -              -              -              -              -       
L                -              -            L->SL'           -       